# Threads Scraper Analysis Stack 

---

## 架構總覽

```
ScrapeCreators API (/v1/threads/*)
        │  HTTP GET + x-api-key
        ▼
┌─ threads_client.py ──────────────────────┐
│  ThreadsClient._request()   (重試 + 錯誤處理)  │
│  ├─ search_posts()    → list[ThreadsPost]     │
│  ├─ get_profile()     → ThreadsProfile        │
│  ├─ get_user_posts()  → list[ThreadsPost]     │
│  ├─ get_post()        → ThreadsPost           │
│  └─ search_users()    → list[dict]            │
│                                               │
│  ThreadsPost.from_api_response()              │
│  └─ 原始 JSON → 正規化 dataclass               │
└───────────────┬───────────────────────┘
                ▼
┌─ keyword_monitor.py ─────────────────────┐
│  KeywordMonitor                               │
│  ├─ search_keyword()  (去重 by post_id)       │
│  ├─ run_round()       (批次搜多關鍵字)         │
│  └─ export_results()  (CSV + JSON → data/)    │
└───────────────┬───────────────────────┘
                ▼
┌─ Notebook (本檔案) ──────────────────────┐
│  載入 → 語言過濾 → 受眾分群 → 品牌分析        │
└──────────────────────────────────────────┘
```

## Pipeline 節點

| # | 節點 | 說明 |
|---|------|------|
| 1 | 環境設置 | 匯入套件、設定 |
| 2 | API 串接 | 初始化 ThreadsClient、驗證連線 |
| 3 | 爬蟲執行 | 用 KeywordMonitor 批次爬取關鍵字貼文 |
| 4 | 資料 Extract | 檢視 API 原始回傳 → dataclass 解析 → 匯出 |
| 5 | 載入 & 過濾 | 讀取 CSV、繁體中文過濾、清洗 |
| 6 | 受眾輪廓 | 高相關貼文、活躍帳號、關鍵字互動 |
| 7 | 受眾分群 | 求推薦 vs 分享心得、品牌/裝備品類分析 |
| 8 | 洞察總結 | 受眾報告 |

## 節點 1：環境設置

匯入所有需要的套件，包括本專案的 `threads_client` 和 `keyword_monitor`。

In [38]:
import os
import sys
from pathlib import Path
DATA_DIR = Path("data")
# 將 files/ 加入模組搜尋路徑
sys.path.insert(0, str(Path("files").resolve()))
print(f"  - 輸出目錄: {DATA_DIR.resolve()}")

  - 輸出目錄: I:\Fnte Workdir\threads-audience-discovery-stack\jp_car_service\data


In [39]:

import json
import re
import time

from collections import Counter
from datetime import datetime

import pandas as pd
import numpy as np

# 匯入自定模組
from threads_client import (
    ThreadsClient, ThreadsPost, ThreadsProfile,
    posts_to_dicts, save_json, save_csv,
)
from keyword_monitor import KeywordMonitor


In [40]:
from dotenv import load_dotenv
load_dotenv(Path(".env"), override=True)

True

## 節點 2：API 串接

初始化 `ThreadsClient`，驗證 API Key 與連線狀態。

**API 架構：**
- 底層使用 `requests.Session` 保持連線
- 所有請求透過 `_request()` 統一處理：自動重試（429/500）、錯誤分類（401/402/400）
- Header 帶 `x-api-key` 認證

In [41]:
# ── 初始化 API Client ──
# API Key 設定
API_KEY = os.environ.get("SCRAPECREATORS_API_KEY")

client = ThreadsClient(api_key=API_KEY, timeout=30, max_retries=3)
print("✔ ThreadsClient 初始化成功")
print(f"  - Base URL: https://api.scrapecreators.com")
print(f"  - Timeout: 30s, Max retries: 3")

# 驗證連線：查詢剩餘額度
try:
    balance = client.get_credit_balance()
    print(f"  - API 額度剩餘: {balance} credits")
except Exception as e:
    print(f"  - ⚠ 額度查詢失敗: {e}")

✔ ThreadsClient 初始化成功
  - Base URL: https://api.scrapecreators.com
  - Timeout: 30s, Max retries: 3
  - API 額度剩餘: 23716 credits


In [42]:
# # ── 逐一測試 5 個 API Endpoint ──
# # 每次呼叫消耗 1 credit

# print("=" * 55)
# print("  測試 5 個 Threads API Endpoints")
# print("=" * 55)

# # ── Endpoint 1: 搜尋貼文（核心爬蟲 endpoint）──
# print("\n── 1. search_posts('AI') ──")
# print("   GET /v1/threads/search?query=AI")
# test_posts = client.search_posts(query="AI")
# print(f"   回傳 {len(test_posts)} 筆貼文")
# if test_posts:
#     p = test_posts[0]
#     print(f"   範例: @{p.username} | {p.like_count} likes")
#     print(f"   Text: {p.text[:80]}...")

# # ── Endpoint 2: 用戶 Profile ──
# print("\n── 2. get_profile('zuck') ──")
# print("   GET /v1/threads/profile?handle=zuck")
# try:
#     profile = client.get_profile("zuck")
#     print(f"   @{profile.username} | {profile.full_name}")
#     print(f"   Followers: {profile.follower_count:,} | Verified: {profile.is_verified}")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 3: 用戶貼文列表 ──
# print("\n── 3. get_user_posts('zuck') ──")
# print("   GET /v1/threads/user/posts?handle=zuck")
# try:
#     user_posts = client.get_user_posts("zuck")
#     print(f"   回傳 {len(user_posts)} 筆貼文")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 4: 搜尋用戶 ──
# print("\n── 4. search_users('tech') ──")
# print("   GET /v1/threads/search/users?query=tech")
# try:
#     users = client.search_users("tech")
#     print(f"   回傳 {len(users)} 位用戶")
#     for u in users[:3]:
#         print(f"   @{u.get('username', '?')} — {u.get('follower_count', '?')} followers")
# except Exception as e:
#     print(f"   Error: {e}")

# # ── Endpoint 5: 單篇貼文詳情 ──
# print("\n── 5. get_post(url) ──")
# print("   GET /v1/threads/post?url=...")
# if test_posts and test_posts[0].permalink:
#     try:
#         single = client.get_post(test_posts[0].permalink)
#         print(f"   @{single.username} | {single.like_count} likes | {single.reply_count} replies")
#     except Exception as e:
#         print(f"   Error: {e}")

# print("\n" + "=" * 55)

## 節點 3：爬蟲執行

使用 `KeywordMonitor` 批次爬取多個關鍵字的貼文。

**執行流程：**
```
KeywordMonitor.run_round(keywords)
  → 逐一呼叫 client.search_posts(keyword)
    → _request() → GET /v1/threads/search?query=...
    → API JSON → ThreadsPost.from_api_response() 逐筆解析
  → seen_ids 去重（跨關鍵字 + 跨輪次）
  → 累積到 all_posts[keyword]
  → export_results() → CSV + JSON
```

In [43]:
# 5 地點 × 3 意圖關鍵字 + 7 機場/景點接送 = 22 credits / 輪
KEYWORDS = [
    # 日本地點：t1：直接包車意圖
    "日本包車", "沖繩包車", "富士山包車", "大阪包車", "北海道包車", "福岡包車",
    # 日本地點：t2：泛旅遊意圖
    "沖繩自由行", "沖繩旅遊", "日本自由行", "日本旅遊",
    "富士山自由行", "富士山旅遊",
    "大阪自由行", "大阪旅遊",
    "北海道自由行", "北海道旅遊",
    "福岡自由行", "福岡旅遊",
    
    # 韓國地點：t1：直接包車意圖
    "韓國包車", "首爾包車", "釜山包車", "濟州島包車",
    # 韓國地點：t2：泛旅遊意圖
    "韓國自由行", "韓國旅遊",
    "首爾自由行", "首爾旅遊",
    "釜山自由行", "釜山旅遊",
    "濟州島自由行", "濟州島旅遊",
    # t1：機場/景點接送
    # "沖繩機場接送", "那霸機場接送", "羽田機場接送",
    # "關西機場接送", "福岡機場接送", "新千歲機場接送",
    # "成田機場接送",
]
PROJECT_TAG = "japan_korea_charter"

In [44]:
# ── 爬蟲模式設定 ──
# MODE = "range"    → 自訂日期區間（手動回測、一次性分析）
# MODE = "schedule" → 模擬 Airflow 排程時段（測試 DAG 邏輯）
# MODE = "day"      → 單日爬取（快速測試）
MODE = "day"

# --- range 模式參數 ---
RANGE_START = "2026-04-02"
RANGE_END   = "2026-04-06"

# --- day 模式參數 ---
DAY_DATE = "2026-04-19"

# --- schedule 模式參數（模擬排程）---
SIMULATE_HOUR = 10  # 模擬時段: 10, 14, 18

# --- 動態輪數設定 ---
USE_ADAPTIVE = False   # True = run_adaptive (自動多輪), False = 固定輪數
FIXED_ROUNDS = 1       # 僅 USE_ADAPTIVE=False 時生效
MIN_ROUNDS = 2         # 僅 USE_ADAPTIVE=True 時生效
MAX_ROUNDS = 4
DUP_THRESHOLD = 0.80   # 當重複率超過此值，且新增數趨緩時，停止後續輪數（僅 adaptive 模式）

# ── get_scrape_params ──
from datetime import timedelta
from zoneinfo import ZoneInfo

TZ_TPE = ZoneInfo("Asia/Taipei")
SCHEDULE_HOURS = [10, 14, 18]

def get_scrape_params(execution_hour: int, ref_date=None):
    """
    根據排程時段計算 API 日期範圍 + post-filter 時間窗口。
    與 fetch_threads_posts.py 中的邏輯一致。
    """
    from datetime import date
    if ref_date is None:
        ref_date = date.today()

    today_str = ref_date.strftime('%Y-%m-%d')
    yesterday_str = (ref_date - timedelta(days=1)).strftime('%Y-%m-%d')
    ref_dt = datetime.combine(ref_date, datetime.min.time()).replace(tzinfo=TZ_TPE)

    closest_hour = min(SCHEDULE_HOURS, key=lambda h: abs(h - execution_hour))

    if closest_hour == 10:
        return {
            "start_date": yesterday_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=0), ref_dt.replace(hour=10)),
            "label": f"10:00 slot | API {yesterday_str}~{today_str}, filter 00:00~10:00",
        }
    else:
        idx = SCHEDULE_HOURS.index(closest_hour)
        prev_hour = SCHEDULE_HOURS[idx - 1]
        return {
            "start_date": today_str,
            "end_date": today_str,
            "time_window": (ref_dt.replace(hour=prev_hour), ref_dt.replace(hour=closest_hour)),
            "label": f"{closest_hour}:00 slot | API {today_str}, filter {prev_hour}:00~{closest_hour}:00",
        }

# ── 根據 MODE 決定參數 ──
if MODE == "schedule":
    params = get_scrape_params(SIMULATE_HOUR)
    START_DATE = params["start_date"]
    END_DATE = params["end_date"]
    TIME_WINDOW = params["time_window"]
    print(f"[schedule] {params['label']}")
    print(f"  time_window: {TIME_WINDOW[0].isoformat()} ~ {TIME_WINDOW[1].isoformat()}")
elif MODE == "day":
    START_DATE = DAY_DATE
    END_DATE = DAY_DATE
    TIME_WINDOW = None
    print(f"[day] {DAY_DATE}")
else:  # range
    START_DATE = RANGE_START
    END_DATE = RANGE_END
    TIME_WINDOW = None
    print(f"[range] {RANGE_START} ~ {RANGE_END}")

print(f"  API params: start_date={START_DATE}, end_date={END_DATE}")
print(f"  adaptive: {'ON' if USE_ADAPTIVE else 'OFF'} (rounds={FIXED_ROUNDS if not USE_ADAPTIVE else f'{MIN_ROUNDS}~{MAX_ROUNDS}'})")
# ── 執行關鍵字爬蟲 ──
monitor = KeywordMonitor(client, output_dir=str(DATA_DIR))

print(f"爬取關鍵字: {len(KEYWORDS)} 組")
print(f"專案標記: {PROJECT_TAG}")
print(f"輸出目錄: {DATA_DIR}")
print(f"預計 API 請求數: {len(KEYWORDS)} 次/輪（每關鍵字 1 次）")
print(f"預計 credits 消耗: 約 {len(KEYWORDS)} credits/輪（以 Credits remaining 實際變化為準）\n")

# ── 執行爬蟲 ──
all_round_stats = []

if USE_ADAPTIVE:
    print(f"[adaptive] min={MIN_ROUNDS}, max={MAX_ROUNDS}, dup_threshold={DUP_THRESHOLD}")
    print("[adaptive] 用途：當新貼文趨緩且重複率升高時，自動停止後續輪數，降低 credits 消耗")
    all_posts, all_round_stats = monitor.run_adaptive(
        KEYWORDS,
        start_date=START_DATE,
        end_date=END_DATE,
        min_rounds=MIN_ROUNDS,
        max_rounds=MAX_ROUNDS,
        dup_threshold=DUP_THRESHOLD,
    )
else:
    for round_num in range(FIXED_ROUNDS):
        print(f"{'='*40} 第 {round_num+1}/{FIXED_ROUNDS} 輪 {'='*40}")
        _results, _stats = monitor.run_round(KEYWORDS, start_date=START_DATE, end_date=END_DATE)
        all_round_stats.append(_stats)
        print(f"  API回傳總筆數={_stats['api_total']}, 新增筆數={_stats['new_count']}, 重複率={_stats['dup_ratio']:.1%}")

# ── 合併所有關鍵字，加上 search_keyword 欄，去重 ──
rows = []
for kw, posts in monitor.all_posts.items():
    for p in posts:
        d = posts_to_dicts([p])[0]
        d['search_keyword'] = kw
        rows.append(d)

df_out = pd.DataFrame(rows).drop_duplicates(subset='post_id', keep='first')

# ── 時間窗口 post-filter（僅 schedule 模式）──
if TIME_WINDOW is not None:
    df_out['timestamp'] = pd.to_datetime(df_out['timestamp'], utc=True)
    start_utc = TIME_WINDOW[0].astimezone(ZoneInfo("UTC"))
    end_utc = TIME_WINDOW[1].astimezone(ZoneInfo("UTC"))
    before = len(df_out)
    df_out = df_out[(df_out['timestamp'] >= start_utc) & (df_out['timestamp'] < end_utc)].copy()
    print(f"\n[time_window filter] {before} -> {len(df_out)} posts")

print(f"\n── 爬取完成 ──")
print(f"  去重後貼文: {len(df_out)} 篇")
print(f"  總輪數: {len(all_round_stats)}")
total_api_results = sum(s.get('api_total', 0) for s in all_round_stats)
estimated_credits = len(KEYWORDS) * len(all_round_stats)
print(f"  API回傳總筆數(各輪加總): {total_api_results}")
print(f"  估算 credits 消耗: 約 {estimated_credits}（以 Credits remaining 差值為準）")

# ── 多輪策略覆蓋率提升評估  ── 
round_new = [s.get('new_count', 0) for s in all_round_stats]
if round_new:
    baseline = round_new[0]
    final_cum = sum(round_new)
    lift_pct = ((final_cum - baseline) / baseline * 100) if baseline > 0 else 0.0

    print(f"  第1輪新增: {baseline}")
    print(f"  多輪累積新增: {final_cum}")
    print(f"  覆蓋率提升(相對第1輪): {lift_pct:.1f}%")
    print("  各輪明細:")
    for i, s in enumerate(all_round_stats, 1):
        print(f"    R{i}: api={s['api_total']}, new={s['new_count']}, dup={s['dup_ratio']:.1%}")
#  ── 

# # ── 原始資料立即儲存 ──
# raw_csv = DATA_DIR / f"threads_{PROJECT_TAG}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# raw_json = raw_csv.with_suffix('.json')
# df_out.to_csv(raw_csv, index=False, encoding='utf-8')
# save_json(df_out.to_dict(orient='records'), str(raw_json))
# print(f"  {raw_csv.name}")
# print(f"  {raw_json.name}")

2026-04-20 09:43:27 [INFO] ─── Search round: 2026-04-20 09:43:27 ───


[day] 2026-04-19
  API params: start_date=2026-04-19, end_date=2026-04-19
  adaptive: OFF (rounds=1)
爬取關鍵字: 30 組
專案標記: japan_korea_charter
輸出目錄: data
預計 API 請求數: 30 次/輪（每關鍵字 1 次）
預計 credits 消耗: 約 30 credits/輪（以 Credits remaining 實際變化為準）

======================================== 第 1/1 輪 ========================================


2026-04-20 09:43:29 [INFO] Credits remaining: 23715
2026-04-20 09:43:29 [INFO]   '日本包車': 9 results, 8 new (total unique: 8)
2026-04-20 09:43:33 [INFO] Credits remaining: 23714
2026-04-20 09:43:33 [INFO]   '沖繩包車': 10 results, 7 new (total unique: 7)
2026-04-20 09:43:36 [INFO] Credits remaining: 23713
2026-04-20 09:43:36 [INFO]   '富士山包車': 10 results, 5 new (total unique: 5)
2026-04-20 09:43:39 [INFO] Credits remaining: 23712
2026-04-20 09:43:39 [INFO]   '大阪包車': 7 results, 5 new (total unique: 5)
2026-04-20 09:43:43 [INFO] Credits remaining: 23711
2026-04-20 09:43:43 [INFO]   '北海道包車': 10 results, 5 new (total unique: 5)
2026-04-20 09:43:46 [INFO] Credits remaining: 23710
2026-04-20 09:43:46 [INFO]   '福岡包車': 10 results, 7 new (total unique: 7)
2026-04-20 09:43:52 [INFO] Credits remaining: 23709
2026-04-20 09:43:52 [INFO]   '沖繩自由行': 10 results, 6 new (total unique: 6)
2026-04-20 09:43:55 [INFO] Credits remaining: 23708
2026-04-20 09:43:55 [INFO]   '沖繩旅遊': 10 results, 9 new (total unique: 9)

  API回傳總筆數=262, 新增筆數=163, 重複率=37.8%

── 爬取完成 ──
  去重後貼文: 163 篇
  總輪數: 1
  API回傳總筆數(各輪加總): 262
  估算 credits 消耗: 約 30（以 Credits remaining 差值為準）
  第1輪新增: 163
  多輪累積新增: 163
  覆蓋率提升(相對第1輪): 0.0%
  各輪明細:
    R1: api=262, new=163, dup=37.8%


In [45]:
df_out

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword
0,3817093491923187325_79949545875,aididigo,False,带台湾客人富士山一日游。真實紀錄在日本包車的工作日常。\n#日本包车一日游 #富士山实况 #...,784,75,12,2,253,False,https://www.threads.net/@aididigo/post/DT5Ck6c...,1769253027,2026-01-24T11:10:27+00:00,日本包車
1,3878468297961391659_63098760682,kay72999,False,目前在台灣已接手2間甲種旅行社，專門幫台灣人客製化出國小包團，2月到現在已經成交150組客人...,42,73,1,0,12,False,https://www.threads.net/@kay72999/post/DXTFl40...,1776569472,2026-04-19T03:31:12+00:00,日本包車
2,3857704898956414802_78982237475,okinawa_travel_lin,False,您好\n我是定居沖繩22年～沖繩島民\n自營包棟民宿 包車 租車\n我來說兩句喔\n其實日本...,64,3,0,0,10,True,https://www.threads.net/@okinawa_travel_lin/po...,1774094282,2026-03-21T11:58:02+00:00,日本包車
3,3878456324498408023_75229251425,tokyo_ezgo,False,看到Toyota Hiace(海獅)的吞吐能力了吧！\n\n滿足大家族一起包車出遊，行李+戰...,97,39,25,0,2,False,https://www.threads.net/@tokyo_ezgo/post/DXTC3...,1776568044,2026-04-19T03:07:24+00:00,日本包車
4,3878667444470484223_63091544361,nininia.ch,False,包車機場接送真的太舒適\n終於不用辛苦拖著行李搭地鐵了\n這次找了日本通包車\n\n日本在地...,5,1,0,0,0,False,https://www.threads.net/@nininia.ch/post/DXTy3...,1776593214,2026-04-19T10:06:54+00:00,日本包車
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,3878876707718108992_37561166172,joyjoytang_,False,濟州島的美咖\n海超美\n\n🍹orrrn,203,1,3,0,98,False,https://www.threads.net/@joyjoytang_/post/DXUi...,1776618158,2026-04-19T17:02:38+00:00,濟州島旅遊
159,3878842961170228684_63125443982,chi___0615,False,去濟州島兩次吃到的食物們🥰,14,6,0,0,8,False,https://www.threads.net/@chi___0615/post/DXUax...,1776614135,2026-04-19T15:55:35+00:00,濟州島旅遊
160,3878649911693147877_67293007737,applelaiapple,False,釜山真的不好玩嗎？\n不少人說高雄很相似\n身為高雄人是否改去濟州島玩啊😂😂,37,51,0,0,3,False,https://www.threads.net/@applelaiapple/post/DX...,1776591122,2026-04-19T09:32:02+00:00,濟州島旅遊
161,3878567747752434810_32702601227,jejutour123,False,韓國濟州島絕佳隱藏小眾景點\n據說一年只開放50還是100天，🈶小夥伴來旅遊五天都沒碰到開放...,74,3,1,0,188,False,https://www.threads.net/@jejutour123/post/DXTc...,1776581327,2026-04-19T06:48:47+00:00,濟州島旅遊


---

## 節點 5：載入資料 & 資料檢視

讀取爬取結果 CSV，執行基礎清洗與資料概覽。

> **語言策略備註：** 本次爬取未使用程式語言過濾。
> 改以關鍵字設計避開日文——使用繁體中文特有字（如「裝備」的「裝」 vs 日文「装」），
> 自然排除多數日文內容。若未來資料出現大量非繁中貼文，可重新啟用語言過濾。

In [46]:
# ── 轉換層 ──
# df_out = pd.read_csv(DATA_DIR / raw_csv.name, encoding='utf-8')

df = df_out.copy()

df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True)

# ── taken_at 轉台灣時間（新增欄位）──
df['taken_at_tpe'] = (
    pd.to_datetime(df['taken_at'], unit='s', utc=True)
      .dt.tz_convert('Asia/Taipei')
      .dt.strftime('%Y-%m-%d %H:%M:%S')
)

# post_date 改以台灣日期為準（讓日期過濾以台灣時區為基準，避免 UTC 跨日誤差）
df['post_date'] = pd.to_datetime(df['taken_at_tpe']).dt.date

df['text_length'] = df['text'].fillna('').str.len()
engagement_cols = ['like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count']
df['total_engagement'] = df[engagement_cols].sum(axis=1)

# 日期範圍過濾（post_date 為台灣日期，與 START_DATE/END_DATE 對齊）
from datetime import date
filter_start = date.fromisoformat(START_DATE)
filter_end = date.fromisoformat(END_DATE)
x_df = df[(df['post_date'] < filter_start) | (df['post_date'] > filter_end)].copy()
df = df[(df['post_date'] >= filter_start) & (df['post_date'] <= filter_end)].copy()
print(f"[{MODE} filter] 篩選台灣日期: {START_DATE}" + (f" ~ {END_DATE}" if START_DATE != END_DATE else ""))

# 標記每篇貼文命中了哪些關鍵字
for kw in KEYWORDS:
    df[f'has_{kw}'] = df['text'].fillna('').str.contains(kw, case=False)
df['matched_keywords'] = df[[f'has_{kw}' for kw in KEYWORDS]].sum(axis=1)

# 意圖分類：文中有包車/租車/司機/接送 → t1，其餘 → t2
df['audience_type'] = np.where(
    df['text'].fillna('').str.contains(r'包車|租車|司機|接送', regex=True),
    't1', 't2'
)

print(f"日期篩選後剩餘總篇數: {len(df)}")
print(f"抓到不在設定日期範圍內的貼文: {len(x_df)} 篇")

print(f"  t1 (包車意圖): {(df['audience_type']=='t1').sum()} 篇")
print(f"  t2 (旅遊意圖): {(df['audience_type']=='t2').sum()} 篇")

# ── 地點驗證：篩出不含任何目標地點的貼文 ──
LOCATIONS = ["日本", "沖繩", "富士山", "大阪", "北海道", "福岡", "韓國", "首爾", "釜山", "濟州島"]
location_pattern = '|'.join(LOCATIONS)
has_location = df['text'].fillna('').str.contains(location_pattern, case=False)
only_car_service = df[~has_location].copy()
df = df[has_location].copy()

print(f"── 地點驗證 ──")
print(f"  含目標地點: {len(df)} 篇")
print(f"  不含目標地點 (only_car_service): {len(only_car_service)} 篇")


# ── 商業文判定（COMMERCIAL_TERMS）──
import re
COMMERCIAL_TERMS = [
    "LINE ID", "歡迎諮詢", "招募", "互追", "漲粉",
    "立即預訂", "車隊", "限時", "超時費", "客製",
    "經營", "急單", "服務品質", "客人", "提前預約",
    "免費幫忙", "免費協助", "為您", "領取", "公司",
    "留言", "傳給你",
]
commercial_pattern = '|'.join(COMMERCIAL_TERMS)

def is_commercial(text: str) -> bool:
    return bool(re.search(commercial_pattern, text or '', flags=re.IGNORECASE))


[day filter] 篩選台灣日期: 2026-04-19
日期篩選後剩餘總篇數: 119
抓到不在設定日期範圍內的貼文: 44 篇
  t1 (包車意圖): 26 篇
  t2 (旅遊意圖): 93 篇
── 地點驗證 ──
  含目標地點: 112 篇
  不含目標地點 (only_car_service): 7 篇


In [47]:
x_df

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,permalink,taken_at,timestamp,search_keyword,taken_at_tpe,post_date,text_length,total_engagement
0,3817093491923187325_79949545875,aididigo,False,带台湾客人富士山一日游。真實紀錄在日本包車的工作日常。\n#日本包车一日游 #富士山实况 #...,784,75,12,2,253,False,https://www.threads.net/@aididigo/post/DT5Ck6c...,1769253027,2026-01-24 11:10:27+00:00,日本包車,2026-01-24 19:10:27,2026-01-24,66,1126
2,3857704898956414802_78982237475,okinawa_travel_lin,False,您好\n我是定居沖繩22年～沖繩島民\n自營包棟民宿 包車 租車\n我來說兩句喔\n其實日本...,64,3,0,0,10,True,https://www.threads.net/@okinawa_travel_lin/po...,1774094282,2026-03-21 11:58:02+00:00,日本包車,2026-03-21 19:58:02,2026-03-21,288,77
9,3777738195581710960_78811101885,japancharter.link,False,🇯🇵 日本接送機 / 日本包車/一價全含\n成田、羽田、關西、中部、札幌新千歲、沖繩那霸\n...,10,45,2,0,32,False,https://www.threads.net/@japancharter.link/pos...,1764561508,2025-12-01 03:58:28+00:00,沖繩包車,2025-12-01 11:58:28,2025-12-01,501,89
10,3846274187615484693_69414422841,yoppyoyo,False,剛2大2小沖繩7天6夜回來，一天一個行程第一天包車去最北邊，海洋館附近飯店玩三天兩夜，後面路...,24,2,0,0,1,True,https://www.threads.net/@yoppyoyo/post/DVgtf7F...,1772731635,2026-03-05 17:27:15+00:00,沖繩包車,2026-03-06 01:27:15,2026-03-06,132,27
11,3877712714948278403_63383650108,ivy.7410,False,請問沖繩下船有包車推薦嗎！還有石桓島！,3,4,0,0,0,False,https://www.threads.net/@ivy.7410/post/DXQZytT...,1776479399,2026-04-18 02:29:59+00:00,沖繩包車,2026-04-18 10:29:59,2026-04-18,19,7
12,3876446058024901311_63283640521,_____bao.312,False,📍5/25-5/28 沖繩旅遊\n想詢問包車推薦🚗\n\n✔ 人數：6大2小（2個安全座椅）...,7,9,0,0,0,False,https://www.threads.net/@_____bao.312/post/DXL...,1776328402,2026-04-16 08:33:22+00:00,沖繩包車,2026-04-16 16:33:22,2026-04-16,109,16
13,3877738709542440263_63395111950,iness_koko,False,有沒有推薦的旅行社呢？\n預計明年一、二月，一家四口+我姐男友\n\n費用大概抓一人5萬左右...,7,11,0,0,0,False,https://www.threads.net/@iness_koko/post/DXQfs...,1776482498,2026-04-18 03:21:38+00:00,沖繩包車,2026-04-18 11:21:38,2026-04-18,130,18
15,3694024663598320998_74715623370,maazitravel_official,True,富士山包車一日遊！邁齊都幫你安排好了，\n直接躺著玩、躺著拍！\n只要一天\n全富士山最美打...,157,559,4,5,343,False,https://www.threads.net/@maazitravel_official/...,1754582077,2025-08-07 15:54:37+00:00,富士山包車,2025-08-07 23:54:37,2025-08-07,162,1068
16,3878031220748260749_63314622919,hsinyiyumi,False,一個人玩富士山3天2夜是可能的嗎？\n目前條件如下：\n1、飯店看得到富士山\n2、所有移動...,30,38,0,0,4,False,https://www.threads.net/@hsinyiyumi/post/DXRiN...,1776517368,2026-04-18 13:02:48+00:00,富士山包車,2026-04-18 21:02:48,2026-04-18,119,72
20,3539500863394177611_71491171967,japan.metour,False,日本🇯🇵全境包車遊｜訂製遊、接送機、周邊遊\n\n日本一直以來都是非常熱門，大家愛去旅遊的地...,21,178,0,0,42,False,https://www.threads.net/@japan.metour/post/DEe...,1736161405,2025-01-06 11:03:25+00:00,大阪包車,2025-01-06 19:03:25,2025-01-06,364,241


In [48]:
# 排除商家文
df['is_commercial'] = df['text'].fillna('').apply(is_commercial)
post_is_commercial = df[df['is_commercial']].copy()
df_clean = df[~df['is_commercial']].copy()

print(f"商家文過濾: {len(df)} → {len(df_clean)} 篇（排除 {len(post_is_commercial)} 篇商家文）")

商家文過濾: 112 → 91 篇（排除 21 篇商家文）


In [49]:
post_is_commercial

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
1,3878468297961391659_63098760682,kay72999,False,目前在台灣已接手2間甲種旅行社，專門幫台灣人客製化出國小包團，2月到現在已經成交150組客人...,42,73,1,0,12,False,...,False,False,False,False,False,False,False,0,t1,True
14,3878836867240735949_71242645493,kanzakitakayuki1394,False,搭郵輪來沖繩，到底是來玩的還是來排隊的？🚢💀\n看看這人龍，光等計程車或接駁車，一個小時就沒...,1,0,0,0,1,False,...,False,False,False,False,False,False,False,2,t1,True
21,3878459399249959604_80717723553,laitravel_official_,False,🇯🇵来旅游JAPAN🇯🇵\n我们是日本国内现地的绿牌公司.承接自由行客製化，專業司機，經驗老...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,True
24,3878646916951835643_63386601006,lineage8573,False,春らしい振袖で素敵でした。\n\n🎏【大阪京都和服外拍，純和服體驗安排，日本一般旅行紀錄，包...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,0,t1,True
46,3878409300243077569_63470858165,p951357460,False,4/12-17去沖繩旅遊六天分享\n\n✈️4月去真的非常舒適，高溫大概27-28度，晚上偏...,54,10,1,0,39,False,...,False,False,False,False,False,False,False,1,t1,True
62,3878771973825003644_66988706697,hyishuan,False,請教廣大的脆友\n想帶媽媽去日本旅遊（京都）但需要帶輪椅（可以行走但無法太久，所以需要帶輪椅...,1,6,0,0,0,False,...,False,False,False,False,False,False,False,1,t2,True
71,3878787228314988060_69899257085,cherry680821,False,大阪免費景點20選\n市區版先給你這篇先收藏\n\n第一次來大阪真的很容易亂走😅\n這20個...,0,0,0,0,2,False,...,False,False,False,False,False,False,False,3,t2,True
75,3878651876456093869_78413960666,minaosakabase,False,京都​宇治旅行避雷！來關西旅遊必看 🙅‍♀️\n​我是大阪房東 Mina，今天分享我常提醒自...,106,7,7,1,125,False,...,False,False,False,False,False,False,False,0,t2,True
85,3878601957435430405_73692190051,ruru.next_trip,False,九州自由行｜初心者攻略\n\nPART 3｜九州飲食文化🍜\n\n「重口味，是九州的生活哲學...,2,0,0,0,3,False,...,False,False,False,False,False,False,False,0,t2,True
97,3878599253174766509_74198169020,dee_h99012,False,​【徵求】韓國包車服務（首爾接機、釜山送機及一日遊）\n​想請問版上是否有推薦的包車司機或車...,5,8,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,True


In [50]:
only_car_service

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國自由行,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type
3,3878456324498408023_75229251425,tokyo_ezgo,False,看到Toyota Hiace(海獅)的吞吐能力了吧！\n\n滿足大家族一起包車出遊，行李+戰...,97,39,25,0,2,False,...,False,False,False,False,False,False,False,False,0,t1
5,3878625596121617698_80489010954,tipsytrip_,False,幫推推,1,1,0,0,0,True,...,False,False,False,False,False,False,False,False,0,t2
36,3878610830534709849_63478156170,chi686889,False,有沒有人認識可以\n屏東包車到桃園的？,41,60,0,0,2,False,...,False,False,False,False,False,False,False,False,0,t1
52,3878819693294415245_71715282143,bbbbb11bbol,False,想問問看有人用過trip的自己配機+酒功能嗎？ 出行上會有問題嗎？\n\n第一次自己規劃自由...,1,13,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t2
59,3878607463508335942_78569616270,steven.wang1224,False,"東京超酷 5天自由行，出發啦！\n\n5/14 新大谷INN雙床房含早餐，每人 $17,90...",0,2,0,0,0,False,...,False,False,False,False,False,False,False,False,0,t2
112,3878408697974838927_71389500693,tingting9496,False,也想預約車包行車⋯請問有懶人包推薦？,1,1,0,0,0,True,...,False,False,False,False,False,False,False,False,0,t2
138,3878717249960930324_73218086811,huei_0203,False,這次去搭了蠻多次uber\n也有拿收據 \n收據金額都跟跳表金額一樣\n也跟系統上預估金額差...,17,9,1,0,47,False,...,False,False,False,False,False,False,False,False,0,t2


In [51]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
4,3878667444470484223_63091544361,nininia.ch,False,包車機場接送真的太舒適\n終於不用辛苦拖著行李搭地鐵了\n這次找了日本通包車\n\n日本在地...,5,1,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
6,3878531688564918950_76543409550,different_life03,False,去大阪玩什麼，8个不一样景点\n\n📍大阪遊學、日本包車私訊我 @different_lif...,18,0,0,1,56,False,...,False,False,False,False,False,False,False,1,t1,False
7,3878647487066094022_68268284362,ddgd_travel,True,櫻花季的倉敷美觀🌸\n總給人一種歲月靜好的感覺₊ ⊹❀\n\n四國/中國地區 包車旅遊也歡迎...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,False
8,3878409485379691322_63469754586,yamocai,False,從那霸下郵輪，在沖繩大概只有6小時⏱️\n打算直接包車🚐\n有沒有推薦「一定要去」的景點？順...,12,9,0,0,1,False,...,False,False,False,False,False,False,False,0,t1,False
17,3878340078037365439_75229251425,tokyo_ezgo,False,遊河口湖，很建議至少安排一個晚上住宿，除了可以悠閒的囊括周邊所有招牌景點，還可以順便遊五湖其...,111,65,22,0,4,False,...,False,False,False,False,False,False,False,0,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,3878704748317277186_63477961671,feng_en_910,False,如果被現實絆倒 那就去一趟濟州島🍊,625,7,13,0,192,False,...,False,False,False,False,False,False,False,0,t2,False
159,3878842961170228684_63125443982,chi___0615,False,去濟州島兩次吃到的食物們🥰,14,6,0,0,8,False,...,False,False,False,False,False,False,False,0,t2,False
160,3878649911693147877_67293007737,applelaiapple,False,釜山真的不好玩嗎？\n不少人說高雄很相似\n身為高雄人是否改去濟州島玩啊😂😂,37,51,0,0,3,False,...,False,False,False,False,False,False,False,0,t2,False
161,3878567747752434810_32702601227,jejutour123,False,韓國濟州島絕佳隱藏小眾景點\n據說一年只開放50還是100天，🈶小夥伴來旅遊五天都沒碰到開放...,74,3,1,0,188,False,...,False,False,False,False,False,False,False,0,t2,False


In [52]:
# ── 資料觀察 ──
print("=== 資料概覽 ===")
print(f"  貼文數: {len(df_clean)}")
print(f"  不重複帳號: {df_clean['username'].nunique()}")
print(f"  時間範圍: {df_clean['post_date'].min():%Y-%m-%d} ~ {df_clean['post_date'].max():%Y-%m-%d}")
print(f"  is_reply 分布: {df_clean['is_reply'].value_counts().to_dict()}")

print(f"\n=== search_keyword 來源分布 ===")
print(df_clean['search_keyword'].value_counts().to_string())

print(f"\n=== 互動指標描述統計 ===")
print(df_clean[engagement_cols + ['total_engagement']].describe().round(1).to_string())

=== 資料概覽 ===
  貼文數: 91
  不重複帳號: 90
  時間範圍: 2026-04-19 ~ 2026-04-19
  is_reply 分布: {False: 90, True: 1}

=== search_keyword 來源分布 ===
search_keyword
濟州島旅遊     9
日本旅遊      7
韓國旅遊      7
日本自由行     6
濟州島自由行    5
釜山旅遊      5
北海道旅遊     5
福岡旅遊      5
沖繩旅遊      5
韓國自由行     4
福岡包車      4
大阪自由行     3
大阪旅遊      3
福岡自由行     3
沖繩自由行     3
富士山包車     3
日本包車      3
首爾包車      2
釜山包車      2
首爾自由行     2
釜山自由行     2
沖繩包車      1
韓國包車      1
首爾旅遊      1

=== 互動指標描述統計 ===
       like_count  reply_count  repost_count  quote_count  reshare_count  total_engagement
count        91.0         91.0          91.0         91.0           91.0              91.0
mean        487.1         23.7          13.4          0.7          274.6             799.4
std        2667.9         49.9          66.1          2.9         1082.8            3802.9
min           0.0          0.0           0.0          0.0            0.0               0.0
25%           5.0          1.0           0.0          0.0            0.0              10.0
5

In [53]:
# # ── 輸出清洗後 CSV/JSON ──
# csv_path = DATA_DIR / f"threads_{PROJECT_TAG}_clean_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
# json_path = csv_path.with_suffix('.json')

# df_clean.to_csv(csv_path, index=False, encoding='utf-8')
# save_json(df_clean.to_dict(orient='records'), str(json_path))

# print(f"✔ {csv_path.name}")
# print(f"✔ {json_path.name}")

In [55]:
# ── 上傳到 Google Sheets ──
import gspread
from google.oauth2.service_account import Credentials

# 篩選欄位（與 vn_visa 對齊；jp_car 不含 post_category）
ouput_df = df_clean[[
    'username', 'taken_at_tpe', 'post_date', 'text',
    'like_count', 'reply_count', 'repost_count', 'quote_count', 'reshare_count',
    'permalink', 'search_keyword', 'audience_type',
]].copy()
ouput_df['post_category'] = ''
ouput_df['scrape_date'] = datetime.now().strftime('%Y-%m-%d')

SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive",
]
CREDS_PATH = Path("ads-de-01-757fc521ef01.json")
SHEET_ID = "1ld3Mt6ZbT1uNSidxYZN94EuIaHL8AIIw2gNZTiFH4Jw"

creds = Credentials.from_service_account_file(CREDS_PATH, scopes=SCOPES)
gc = gspread.authorize(creds)

sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1

# ── 安全寫入：明確定位最後一行，不依賴 append_rows 的表格偵測 ──
existing = ws.get_all_values()
next_row = len(existing) + 1

if not existing or not any(str(cell).strip() for cell in existing[0]):
    ws.update(range_name='A1', values=[ouput_df.columns.tolist()], value_input_option="RAW")
    next_row = 2
    print(f"[init] 已寫入標題列")

data_rows = ouput_df.astype(str).values.tolist()

if data_rows:
    required_rows = next_row + len(data_rows) - 1
    required_cols = len(ouput_df.columns)
    target_rows = max(ws.row_count, required_rows)
    target_cols = max(ws.col_count, required_cols)

    if target_rows > ws.row_count or target_cols > ws.col_count:
        ws.resize(rows=target_rows, cols=target_cols)
        print(f"[resize] Sheet grid 已調整為 rows={target_rows}, cols={target_cols}")

    ws.update(range_name=f'A{next_row}', values=data_rows, value_input_option="USER_ENTERED")

    print(f"[OK] 已寫入 {len(data_rows)} 筆到 A{next_row}~A{required_rows}")
    print(f"  Sheet 目前共 {required_rows} 行（含標題）")
    print(f"  https://docs.google.com/spreadsheets/d/{SHEET_ID}")
else:
    print("[skip] ouput_df 無資料，略過 Google Sheets 寫入")

[resize] Sheet grid 已調整為 rows=1336, cols=46
[OK] 已寫入 91 筆到 A1246~A1336
  Sheet 目前共 1336 行（含標題）
  https://docs.google.com/spreadsheets/d/1ld3Mt6ZbT1uNSidxYZN94EuIaHL8AIIw2gNZTiFH4Jw


In [57]:
df_clean

,post_id,username,is_verified,text,like_count,reply_count,repost_count,quote_count,reshare_count,is_reply,...,has_韓國旅遊,has_首爾自由行,has_首爾旅遊,has_釜山自由行,has_釜山旅遊,has_濟州島自由行,has_濟州島旅遊,matched_keywords,audience_type,is_commercial
4,3878667444470484223_63091544361,nininia.ch,False,包車機場接送真的太舒適\n終於不用辛苦拖著行李搭地鐵了\n這次找了日本通包車\n\n日本在地...,5,1,0,0,0,False,...,False,False,False,False,False,False,False,1,t1,False
6,3878531688564918950_76543409550,different_life03,False,去大阪玩什麼，8个不一样景点\n\n📍大阪遊學、日本包車私訊我 @different_lif...,18,0,0,1,56,False,...,False,False,False,False,False,False,False,1,t1,False
7,3878647487066094022_68268284362,ddgd_travel,True,櫻花季的倉敷美觀🌸\n總給人一種歲月靜好的感覺₊ ⊹❀\n\n四國/中國地區 包車旅遊也歡迎...,0,0,0,0,0,False,...,False,False,False,False,False,False,False,2,t1,False
8,3878409485379691322_63469754586,yamocai,False,從那霸下郵輪，在沖繩大概只有6小時⏱️\n打算直接包車🚐\n有沒有推薦「一定要去」的景點？順...,12,9,0,0,1,False,...,False,False,False,False,False,False,False,0,t1,False
17,3878340078037365439_75229251425,tokyo_ezgo,False,遊河口湖，很建議至少安排一個晚上住宿，除了可以悠閒的囊括周邊所有招牌景點，還可以順便遊五湖其...,111,65,22,0,4,False,...,False,False,False,False,False,False,False,0,t1,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
157,3878704748317277186_63477961671,feng_en_910,False,如果被現實絆倒 那就去一趟濟州島🍊,625,7,13,0,192,False,...,False,False,False,False,False,False,False,0,t2,False
159,3878842961170228684_63125443982,chi___0615,False,去濟州島兩次吃到的食物們🥰,14,6,0,0,8,False,...,False,False,False,False,False,False,False,0,t2,False
160,3878649911693147877_67293007737,applelaiapple,False,釜山真的不好玩嗎？\n不少人說高雄很相似\n身為高雄人是否改去濟州島玩啊😂😂,37,51,0,0,3,False,...,False,False,False,False,False,False,False,0,t2,False
161,3878567747752434810_32702601227,jejutour123,False,韓國濟州島絕佳隱藏小眾景點\n據說一年只開放50還是100天，🈶小夥伴來旅遊五天都沒碰到開放...,74,3,1,0,188,False,...,False,False,False,False,False,False,False,0,t2,False


In [59]:
backup_csv_path = DATA_DIR / f"threads_car_service_post_{(datetime.now() - timedelta(days=1)).strftime('%Y%m%d')}.csv"
ouput_df.to_csv(backup_csv_path, index=False, encoding='utf-8-sig')
print(f"[OK] 已備份 CSV: {backup_csv_path}")

[OK] 已備份 CSV: data\threads_car_service_post_20260419.csv
